PHASE 1 - DATA PREPARATION

IACOFI 2023 (Bank of Italy - Survey on the financial literacy and the
financial skills of the Italian adult population)

This script:
  1. Loads the raw dataset and recodes special non-response codes to NaN
  2. Builds a two-level "Digital Financial Exclusion" indicator
     - Level 0: no internet access (qd14 == 0)
     - Level 1: internet access but low/no use of digital financial tools
  3. Builds a Digital Adoption Score (continuous, for users with internet)
  4. Builds an objective Financial Literacy Score from the QK block
  5. Cleans the key socio-demographic variables
  6. Saves a clean, analysis-ready dataset

In [ ]:
import pandas as pd
import numpy as np

RAW_PATH = "Database_ENG.csv"
OUT_PATH = "cleaned_dataset.csv"

1. LOAD DATA

In [ ]:
df = pd.read_csv(RAW_PATH)
print(f"Raw dataset: {df.shape[0]} respondents, {df.shape[1]} variables")

Raw dataset: 4862 respondents, 219 variables


2. RECODE SPECIAL NON-RESPONSE CODES TO NaN

In [ ]:
# -97 = "don't know", -98 = "not applicable / none", -99 = "refusal",
# -999 = "irrelevant answer" (used in qk3, qk4, qk5, qk6)
SPECIAL_CODES = [-97, -98, -99, -999]

# Apply the recoding to every numeric column EXCEPT the survey weight,
# which never contains special codes.
cols_to_recode = [c for c in df.columns if c != "wght"]
for c in cols_to_recode:
    df[c] = df[c].replace(SPECIAL_CODES, np.nan)

print("Special non-response codes (-97, -98, -99, -999) recoded to NaN.")

Special non-response codes (-97, -98, -99, -999) recoded to NaN.


3. SOCIO-DEMOGRAPHIC VARIABLES - CLEANING AND LABELLING

In [ ]:
# All derived columns are first collected in a dict and added to the
# DataFrame with a single pd.concat, to avoid the repeated column-by-column
# insertions that trigger pandas' "highly fragmented DataFrame" warning.
new_cols = {}

# --- Gender (qd1) ---
new_cols["gender"] = df["qd1"].map({0: "Male", 1: "Female"})

# --- Age (qd7) ---
# 219 respondents refused to give their exact age (qd7 -> NaN after recoding)
# but provided an age bracket in qd7_a. We recover age as a bracket variable
# so that these respondents are not dropped from the descriptive analysis.
AGE_BRACKET_MAP = {
    1: "18-19", 2: "20-29", 3: "30-39", 4: "40-49",
    5: "50-59", 6: "60-69", 7: "70-79"
}

def age_to_bracket(age):
    if pd.isna(age):
        return np.nan
    if age < 20:
        return "18-19"
    elif age < 30:
        return "20-29"
    elif age < 40:
        return "30-39"
    elif age < 50:
        return "40-49"
    elif age < 60:
        return "50-59"
    elif age < 70:
        return "60-69"
    else:
        return "70-79"

new_cols["age"] = df["qd7"]
age_bracket = df["qd7"].apply(age_to_bracket)
# fill in the bracket for respondents who refused the exact age (qd7_a)
mask_missing_age = age_bracket.isna() & df["qd7_a"].notna()
age_bracket.loc[mask_missing_age] = df.loc[mask_missing_age, "qd7_a"].map(AGE_BRACKET_MAP)
new_cols["age_bracket"] = age_bracket

# --- Macro-region (qd2) ---
REGION_MAP = {1: "North-West", 2: "North-East", 3: "Centre", 4: "South", 5: "Islands"}
new_cols["region"] = df["qd2"].map(REGION_MAP)

# --- Type of municipality (qd3) ---
MUNICIPALITY_MAP = {
    1: "Up to 2,000 inhabitants",
    2: "2,001-10,000 inhabitants",
    3: "10,001-50,000 inhabitants",
    4: "50,001-250,000 inhabitants",
    5: "More than 250,000 inhabitants",
}
new_cols["municipality_size"] = df["qd3"].map(MUNICIPALITY_MAP)

# --- Education (qd9) ---
# Collapsed into a 4-level ordinal scale for tractability in later modelling
EDU_MAP = {
    1: "Low",   # none / primary
    2: "Low",   # lower secondary
    3: "Medium",  # vocational, no diploma
    4: "Medium",  # vocational diploma
    5: "Medium",  # upper secondary diploma
    6: "Medium",  # post-secondary non-tertiary
    7: "High",  # bachelor's degree / equivalent
    8: "High",  # master's degree
    9: "High",  # postgraduate (specialisation, PhD)
    10: "High", # other tertiary
}
education = df["qd9"].map(EDU_MAP)
new_cols["education"] = education.astype(
    pd.CategoricalDtype(categories=["Low", "Medium", "High"], ordered=True)
)

# --- Employment status (qd10) ---
EMPLOYMENT_MAP = {
    1: "Self-employed",
    2: "Employee",
    3: "Employee (temporary)",
    4: "Not employed - looking for work",
    5: "Not employed - not looking for work",
    6: "Retired",
    7: "Unable to work",
    8: "Homemaker",
    9: "Student",
}
new_cols["employment"] = df["qd10"].map(EMPLOYMENT_MAP)

# --- Household income bracket (qd13) ---
# NOTE: ~31% missing (refusal/don't know). We keep a "Not declared" category
# instead of dropping these respondents, since dropping them would remove
# almost a third of the sample.
INCOME_MAP = {1: "Up to 1,750 EUR", 2: "1,751-2,900 EUR", 3: "Over 2,900 EUR"}
income_bracket = df["qd13"].map(INCOME_MAP).astype("object")
new_cols["income_bracket"] = income_bracket.fillna("Not declared")

# --- Internet access (qd14) ---
new_cols["internet_access"] = df["qd14"].map({0: "No", 1: "Yes"})

# --- Survey weight ---
new_cols["weight"] = df["wght"]

# Add all new columns to df in a single operation
df = pd.concat([df, pd.DataFrame(new_cols, index=df.index)], axis=1)

4. DIGITAL ADOPTION: TWO-LEVEL FRAMEWORK

In [ ]:
# Level 0 - Basic digital divide: no internet access at all.
#   These respondents are, by construction, excluded from every digital
#   financial service and have NaN on the QP8/QP9 blocks.
#
# Level 1 - Digital financial exclusion (conditional on internet access):
#   built from QP8 (ever performed financial operations fully online,
#   binary 0/1 items) and QP9 (frequency, last 12 months, of digital
#   financial activities, scale 1='never' to 4='very often').

qp8_items = ["qp8_1", "qp8_2", "qp8_3", "qp8_4", "qp8_5"]
qp9_items = ["qp9_1", "qp9_3", "qp9_4", "qp9_5", "qp9_6", "qp9_7", "qp9_10"]

# Rescale QP9 (1-4) to a 0-1 range so it is comparable with QP8 (0/1)
qp9_rescaled = (df[qp9_items] - 1) / 3

# Digital Adoption Score (DAS): average across QP8 (binary) and rescaled QP9
# items, expressed on a 0-100 scale. Only defined for internet users.
adoption_items = pd.concat([df[qp8_items], qp9_rescaled], axis=1)

adoption_extra = pd.DataFrame({
    "digital_adoption_score": adoption_items.mean(axis=1, skipna=True) * 100,
    "digital_adoption_n_valid": adoption_items.notna().sum(axis=1),
}, index=df.index)
df = pd.concat([df, adoption_extra], axis=1)

# --- Digital exclusion status (3 categories) ---
# - "No internet access"        -> Level 0
# - "Internet, low digital use" -> Level 1 (adoption score in bottom tercile
#                                   among internet users, computed below)
# - "Internet, digital adopter" -> the rest of internet users
exclusion_status = pd.Series(pd.NA, index=df.index, dtype="object")
exclusion_status.loc[df["internet_access"] == "No"] = "No internet access"

internet_users = df["internet_access"] == "Yes"
# tercile threshold computed on internet users only, weighted
das_internet = df.loc[internet_users, "digital_adoption_score"].dropna()
w_internet = df.loc[das_internet.index, "weight"]

# weighted tercile (33rd percentile) as the cut-off for "low digital use"
sorted_idx = das_internet.sort_values().index
cum_w = w_internet.loc[sorted_idx].cumsum()
threshold_idx = (cum_w >= cum_w.iloc[-1] / 3).idxmax()
das_threshold = das_internet.loc[threshold_idx]

exclusion_status.loc[internet_users & (df["digital_adoption_score"] <= das_threshold)] = \
    "Internet, low digital use"
exclusion_status.loc[internet_users & (df["digital_adoption_score"] > das_threshold)] = \
    "Internet, digital adopter"

# Binary outcome for later modelling:
# 1 = "digitally excluded from finance" (no internet OR internet+low use)
# 0 = "digital adopter"
exclusion_binary = np.where(
    exclusion_status.isin(["No internet access", "Internet, low digital use"]),
    1,
    np.where(exclusion_status == "Internet, digital adopter", 0, np.nan),
)

exclusion_extra = pd.DataFrame({
    "digital_exclusion_status": exclusion_status,
    "digital_exclusion_binary": exclusion_binary,
}, index=df.index)
df = pd.concat([df, exclusion_extra], axis=1)

print(f"\nWeighted DAS tercile threshold (low-use cut-off): {das_threshold:.2f}")
print(df["digital_exclusion_status"].value_counts(dropna=False))


Weighted DAS tercile threshold (low-use cut-off): 13.89
digital_exclusion_status
Internet, digital adopter    2954
Internet, low digital use    1430
No internet access            415
<NA>                           63
Name: count, dtype: int64


5. FINANCIAL LITERACY SCORE (OECD/INFE-STYLE OBJECTIVE SCORE)

In [ ]:
# Built from 7 objective knowledge items: qk3, qk5, qk6, qk7_1, qk7_2,
# qk7_3, qk10. Each item is recoded to 1 (correct) / 0 (incorrect or
# don't know) and summed, giving a score from 0 to 7.
#
# Correct answers:
#  - qk3  (inflation, multiple choice)        -> correct = 3
#  - qk5  (simple interest, open numeric)      -> correct = 102
#  - qk6  (compound interest, multiple choice) -> correct = 1
#  - qk7_1 (risk/return statement, T/F)        -> correct = 1
#  - qk7_2 (inflation statement, T/F)          -> correct = 1
#  - qk7_3 (diversification statement, T/F)    -> correct = 1
#  - qk10 (mortgage statement, T/F)            -> correct = 1
#
# "Don't know" / missing values are scored as incorrect (0), following the
# standard OECD/INFE approach: not knowing the correct answer is treated
# the same as answering incorrectly for the purpose of the literacy score.

def score_qk3(x):
    return 1 if x == 3 else 0

def score_qk5(x):
    return 1 if x == 102 else 0

def score_qk6(x):
    return 1 if x == 1 else 0

def score_binary_correct1(x):
    return 1 if x == 1 else 0

lit_df = pd.DataFrame({
    "lit_qk3": df["qk3"].apply(score_qk3),
    "lit_qk5": df["qk5"].apply(score_qk5),
    "lit_qk6": df["qk6"].apply(score_qk6),
    "lit_qk7_1": df["qk7_1"].apply(score_binary_correct1),
    "lit_qk7_2": df["qk7_2"].apply(score_binary_correct1),
    "lit_qk7_3": df["qk7_3"].apply(score_binary_correct1),
    "lit_qk10": df["qk10"].apply(score_binary_correct1),
}, index=df.index)

lit_items = list(lit_df.columns)
lit_df["financial_literacy_score"] = lit_df[lit_items].sum(axis=1)  # 0-7

# Self-assessed financial knowledge (qk1, 1-5 scale), kept separately as it
# is a subjective measure and not part of the objective score.
lit_df["self_assessed_knowledge"] = df["qk1"]

df = pd.concat([df, lit_df], axis=1)

print("\nFinancial literacy score (0-7) distribution:")
print(df["financial_literacy_score"].value_counts().sort_index())


Financial literacy score (0-7) distribution:
financial_literacy_score
0     338
1     329
2     538
3     698
4     962
5    1076
6     627
7     294
Name: count, dtype: int64


6. FINAL VARIABLE SELECTION AND EXPORT

In [ ]:
keep_cols = [
    "weight",
    "gender", "age", "age_bracket", "region", "municipality_size",
    "education", "employment", "income_bracket", "internet_access",
    "digital_adoption_score", "digital_adoption_n_valid",
    "digital_exclusion_status", "digital_exclusion_binary",
    "financial_literacy_score", "self_assessed_knowledge",
]

clean_df = df[keep_cols].copy()
clean_df.to_csv(OUT_PATH, index=False)

print(f"\nClean dataset saved to '{OUT_PATH}': {clean_df.shape[0]} rows, {clean_df.shape[1]} columns")



Clean dataset saved to 'cleaned_dataset.csv': 4862 rows, 16 columns


7. QUICK QUALITY CHECKS (printed for the report)

In [ ]:
print("\n--- Missingness in key variables ---")
print(clean_df.isna().mean().round(3) * 100)

print("\n--- Weighted summary: digital exclusion status ---")
ws = clean_df.groupby("digital_exclusion_status")["weight"].sum()
print((ws / ws.sum() * 100).round(1))

print("\n--- Weighted mean financial literacy score by digital exclusion status ---")
for status in clean_df["digital_exclusion_status"].dropna().unique():
    sub = clean_df[clean_df["digital_exclusion_status"] == status]
    wmean = np.average(sub["financial_literacy_score"], weights=sub["weight"])
    print(f"{status}: {wmean:.2f}")


--- Missingness in key variables ---
weight                      0.0
gender                      0.0
age                         4.5
age_bracket                 0.0
region                      0.0
municipality_size           0.0
education                   0.0
employment                  0.4
income_bracket              0.0
internet_access             0.4
digital_adoption_score      9.8
digital_adoption_n_valid    0.0
digital_exclusion_status    1.3
digital_exclusion_binary    1.3
financial_literacy_score    0.0
self_assessed_knowledge     8.4
dtype: float64

--- Weighted summary: digital exclusion status ---
digital_exclusion_status
Internet, digital adopter    57.0
Internet, low digital use    32.3
No internet access           10.7
Name: weight, dtype: float64

--- Weighted mean financial literacy score by digital exclusion status ---
Internet, digital adopter: 4.05
Internet, low digital use: 3.35
No internet access: 2.42
